# songo-model-stockfish-for-google-collab - All In One

Notebook unique pour:
- monter Google Drive
- cloner ou mettre a jour le code depuis GitHub
- installer les dependances
- lancer la generation de dataset
- construire le dataset final
- entrainer le modele
- evaluer le modele
- benchmarker le modele
- reprendre un job interrompu avec le meme `job_id`


## 1. Monter Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuration du projet

In [ ]:
GIT_REPO_URL = 'https://github.com/TON-COMPTE/songo-model-stockfish-for-google-collab.git'
GIT_BRANCH = 'main'
PROJECT_NAME = 'songo-model-stockfish-for-google-collab'
DRIVE_ROOT = '/content/drive/MyDrive/songo-stockfish'
WORKTREE = f'/content/{PROJECT_NAME}'

print('GIT_REPO_URL =', GIT_REPO_URL)
print('GIT_BRANCH   =', GIT_BRANCH)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('WORKTREE     =', WORKTREE)


## 3. Initialiser Drive

In [ ]:
import os

for path in [
    f'{DRIVE_ROOT}/code',
    f'{DRIVE_ROOT}/code_snapshots',
    f'{DRIVE_ROOT}/data/raw',
    f'{DRIVE_ROOT}/data/raw_colab_pro',
    f'{DRIVE_ROOT}/data/raw_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/sampled',
    f'{DRIVE_ROOT}/data/sampled_colab_pro',
    f'{DRIVE_ROOT}/data/sampled_full_matrix_colab_pro',
    f'{DRIVE_ROOT}/data/datasets',
    f'{DRIVE_ROOT}/jobs',
    f'{DRIVE_ROOT}/models/checkpoints',
    f'{DRIVE_ROOT}/models/final',
    f'{DRIVE_ROOT}/models/promoted/best',
    f'{DRIVE_ROOT}/models/lineage',
    f'{DRIVE_ROOT}/logs',
    f'{DRIVE_ROOT}/reports/evaluations',
    f'{DRIVE_ROOT}/reports/benchmarks',
    f'{DRIVE_ROOT}/exports',
]:
    os.makedirs(path, exist_ok=True)

print('Drive layout ready')


## 4. Cloner ou mettre a jour le repo

In [ ]:
import os
import shutil
import subprocess

if not os.path.exists(os.path.join(WORKTREE, '.git')):
    if os.path.exists(WORKTREE):
        shutil.rmtree(WORKTREE)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO_URL, WORKTREE], check=True)
else:
    subprocess.run(['git', '-C', WORKTREE, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', WORKTREE, 'pull', '--ff-only', 'origin', GIT_BRANCH], check=True)

print('Worktree ready')


## 5. Installer les dependances

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', f'{WORKTREE}/requirements.txt'], check=True)
print('Environment ready in Colab runtime')


## 6. Choisir les configs et job ids

Le pipeline dataset supporte maintenant plusieurs modes de generation:
- `benchmatch`: generation longue a partir des matchs de reference
- `clone_existing`: creation d'une nouvelle source de dataset a partir d'un corpus deja existant
- `derive_existing`: creation d'une nouvelle variante filtree a partir d'un corpus existant
- `merge_existing`: fusion de plusieurs sources deja versionnees en une nouvelle grande source

Ensuite `dataset-build` peut choisir explicitement quelle source de dataset il enrichit.
Les originaux restent conserves, car chaque source et chaque dataset construit sont versionnes separement.
Tu peux aussi forcer un nouvel identifiant de dataset construit avec `DATASET_BUILD_ID_OVERRIDE`.


Pour comparer proprement deux familles de modeles, garde le meme dataset et change seulement `TRAIN_CONFIG` et les `JOB_ID`.

Exemple recommande:
- stable: `TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.yaml'`
- residual: `TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.residual.yaml'`

Important:
- utilise des `TRAIN_JOB_ID`, `EVALUATION_JOB_ID` et `BENCHMARK_JOB_ID` differents entre stable et residual
- ne reutilise pas le meme `TRAIN_JOB_ID` pour deux architectures differentes


In [ ]:
DATASET_GENERATE_CONFIG = 'config/dataset_generation.full_matrix.colab_pro.yaml'
DATASET_BUILD_CONFIG = 'config/dataset_build.full_matrix.colab_pro.yaml'
DATASET_MERGE_FINAL_CONFIG = 'config/dataset_merge_final.colab_pro.yaml'
TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.yaml'
# Variante experimentale possible:
# TRAIN_CONFIG = 'config/train.full_matrix.colab_pro.residual.yaml'
EVALUATION_CONFIG = 'config/evaluation.full_matrix.colab_pro.yaml'
BENCHMARK_CONFIG = 'config/benchmark.colab_pro.yaml'

DATASET_GENERATE_JOB_ID = 'dataset_full_matrix_colab_pro_1m_001'
DATASET_BUILD_JOB_ID = 'dataset_build_full_matrix_colab_pro_1m_001'
DATASET_MERGE_FINAL_JOB_ID = 'dataset_merge_final_colab_pro_001'
TRAIN_JOB_ID = 'train_colab_pro_stable_001'
EVALUATION_JOB_ID = 'eval_colab_pro_stable_001'
BENCHMARK_JOB_ID = 'benchmark_colab_pro_stable_001'


In [ ]:
DATASET_GENERATION_MODE = 'benchmatch'
DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro'
SOURCE_DATASET_ID_FOR_GENERATION = ''
SOURCE_DATASET_IDS_FOR_GENERATION = []
DERIVATION_STRATEGY = 'unique_positions'
TARGET_SAMPLES = 1000000
SOURCE_DATASET_ID_FOR_BUILD = DATASET_SOURCE_ID
DATASET_BUILD_ID_OVERRIDE = ''
TARGET_LABELED_SAMPLES = 1000000
MERGED_FINAL_DATASET_ID = 'dataset_merged_final_colab_pro_insane_1m'
MERGE_ALL_BUILT_DATASETS = True
MERGE_SOURCE_DATASET_IDS = []
COMPARE_DATASET_ID_A = ''
COMPARE_DATASET_ID_B = ''

DATASET_GENERATE_EXTRA_ARGS = f"--generation-mode {DATASET_GENERATION_MODE} --dataset-source-id {DATASET_SOURCE_ID}"
if SOURCE_DATASET_ID_FOR_GENERATION:
    DATASET_GENERATE_EXTRA_ARGS += f" --source-dataset-id {SOURCE_DATASET_ID_FOR_GENERATION}"
if DATASET_GENERATION_MODE == 'merge_existing' and SOURCE_DATASET_IDS_FOR_GENERATION:
    joined_source_ids = ' '.join(SOURCE_DATASET_IDS_FOR_GENERATION)
    DATASET_GENERATE_EXTRA_ARGS += f" --source-dataset-ids {joined_source_ids}"
if DATASET_GENERATION_MODE == 'derive_existing':
    DATASET_GENERATE_EXTRA_ARGS += f" --derivation-strategy {DERIVATION_STRATEGY}"
if TARGET_SAMPLES:
    DATASET_GENERATE_EXTRA_ARGS += f" --target-samples {TARGET_SAMPLES}"

DATASET_BUILD_EXTRA_ARGS = ''
if SOURCE_DATASET_ID_FOR_BUILD:
    DATASET_BUILD_EXTRA_ARGS += f" --source-dataset-id {SOURCE_DATASET_ID_FOR_BUILD}"
if DATASET_BUILD_ID_OVERRIDE:
    DATASET_BUILD_EXTRA_ARGS += f" --dataset-id-override {DATASET_BUILD_ID_OVERRIDE}"
if TARGET_LABELED_SAMPLES:
    DATASET_BUILD_EXTRA_ARGS += f" --target-labeled-samples {TARGET_LABELED_SAMPLES}"

DATASET_MERGE_FINAL_EXTRA_ARGS = f"--dataset-id {MERGED_FINAL_DATASET_ID}"
if MERGE_ALL_BUILT_DATASETS:
    DATASET_MERGE_FINAL_EXTRA_ARGS += ' --include-all-built'
elif MERGE_SOURCE_DATASET_IDS:
    joined_ids = ' '.join(MERGE_SOURCE_DATASET_IDS)
    DATASET_MERGE_FINAL_EXTRA_ARGS += f" --source-dataset-ids {joined_ids}"

print('DATASET_GENERATION_MODE        =', DATASET_GENERATION_MODE)
print('DATASET_SOURCE_ID              =', DATASET_SOURCE_ID)
print('SOURCE_DATASET_ID_FOR_GENERATION =', SOURCE_DATASET_ID_FOR_GENERATION)
print('SOURCE_DATASET_IDS_FOR_GENERATION =', SOURCE_DATASET_IDS_FOR_GENERATION)
print('DERIVATION_STRATEGY            =', DERIVATION_STRATEGY)
print('TARGET_SAMPLES                =', TARGET_SAMPLES)
print('SOURCE_DATASET_ID_FOR_BUILD    =', SOURCE_DATASET_ID_FOR_BUILD)
print('DATASET_BUILD_ID_OVERRIDE      =', DATASET_BUILD_ID_OVERRIDE)
print('TARGET_LABELED_SAMPLES        =', TARGET_LABELED_SAMPLES)
print('MERGED_FINAL_DATASET_ID       =', MERGED_FINAL_DATASET_ID)
print('MERGE_ALL_BUILT_DATASETS      =', MERGE_ALL_BUILT_DATASETS)
print('MERGE_SOURCE_DATASET_IDS      =', MERGE_SOURCE_DATASET_IDS)
print('COMPARE_DATASET_ID_A          =', COMPARE_DATASET_ID_A)
print('COMPARE_DATASET_ID_B          =', COMPARE_DATASET_ID_B)
print('DATASET_GENERATE_EXTRA_ARGS    =', DATASET_GENERATE_EXTRA_ARGS)
print('DATASET_BUILD_EXTRA_ARGS       =', DATASET_BUILD_EXTRA_ARGS)
print('DATASET_MERGE_FINAL_EXTRA_ARGS =', DATASET_MERGE_FINAL_EXTRA_ARGS)


Presets utiles:

- Corpus long de reference par matchs:
  - `DATASET_GENERATION_MODE = 'benchmatch'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = ''`

- Nouvelle source rapide a partir du corpus 1M deja existant:
  - `DATASET_GENERATION_MODE = 'clone_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_variant_a'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = 'sampled_full_matrix_colab_pro'`

- Variante derivee sans relancer les matchs:
  - `DATASET_GENERATION_MODE = 'derive_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_unique'`
  - `SOURCE_DATASET_ID_FOR_GENERATION = 'sampled_full_matrix_colab_pro'`
  - `DERIVATION_STRATEGY = 'unique_positions'`
  - `TARGET_SAMPLES = 2000000`

- Grande source fusionnee a partir de plusieurs sources deja versionnees:
  - `DATASET_GENERATION_MODE = 'merge_existing'`
  - `DATASET_SOURCE_ID = 'sampled_full_matrix_colab_pro_giant'`
  - `SOURCE_DATASET_IDS_FOR_GENERATION = ['sampled_full_matrix_colab_pro', 'sampled_full_matrix_colab_pro_unique']`
  - `TARGET_SAMPLES = 2000000`

- Build d'un nouveau dataset final a partir d'une source existante:
  - `SOURCE_DATASET_ID_FOR_BUILD = 'sampled_full_matrix_colab_pro_variant_a'`
  - `DATASET_BUILD_ID_OVERRIDE = 'dataset_v3_variant_a_insane_2m'`
  - `TARGET_LABELED_SAMPLES = 2000000`

- Fusionner tous les datasets finaux existants en un nouveau dataset final versionne:
  - `MERGED_FINAL_DATASET_ID = 'dataset_merged_final_colab_pro_insane_1m'`
  - `MERGE_ALL_BUILT_DATASETS = True`

- Comparer deux datasets finaux existants:
  - `COMPARE_DATASET_ID_A = 'dataset_v2_full_matrix_colab_pro_insane_1m'`
  - `COMPARE_DATASET_ID_B = 'dataset_merged_final_colab_pro_insane_1m'`


Convention recommandee de nommage:
- sources: `sampled_full_matrix_colab_pro_<variante>`
- datasets finaux: `dataset_vN_full_matrix_colab_pro_<variante>_insane_1m`

Evite de reutiliser un identifiant pour un contenu different.


In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    print('Dataset sources:')
    for item in registry.get('dataset_sources', [])[-10:]:
        print('-', item.get('dataset_source_id'), '| mode=', item.get('source_mode'), '| parent=', item.get('source_dataset_id', ''), '| parents=', item.get('source_dataset_ids', []), '| samples=', item.get('sampled_positions'))
    print('\nBuilt datasets:')
    for item in registry.get('built_datasets', [])[-10:]:
        print('-', item.get('dataset_id'), '| build_mode=', item.get('build_mode', 'teacher_label'), '| source=', item.get('source_dataset_id'), '| teacher=', f"{item.get('teacher_engine')}:{item.get('teacher_level')}", '| labeled=', item.get('labeled_samples'))
else:
    print('Aucun dataset_registry.json trouve pour le moment')


Tu peux aussi utiliser directement la CLI `dataset-list` pour relire le registre avec le meme comportement que hors notebook.

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-list --config $DATASET_GENERATE_CONFIG --kind all"

## 7. Lancer la generation du dataset

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-generate --config $DATASET_GENERATE_CONFIG --job-id $DATASET_GENERATE_JOB_ID $DATASET_GENERATE_EXTRA_ARGS"

Lecture rapide du dernier resume `dataset-generate`, avec affichage detaille du `source_breakdown` quand le mode est `merge_existing`.

In [ ]:
from pathlib import Path
import json
import re

jobs_root = Path(DRIVE_ROOT) / 'jobs'
requested_job_id = DATASET_GENERATE_JOB_ID

match = re.match(r'^(.*?)(\d+)$', requested_job_id)
if match:
    prefix = match.group(1)
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and re.match(rf'^{re.escape(prefix)}\d+$', path.name)]
else:
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and path.name.startswith(requested_job_id)]

if not candidates:
    print('Aucun job dataset-generate trouve pour', requested_job_id)
else:
    latest_job_dir = sorted(candidates, key=lambda path: path.stat().st_mtime)[-1]
    summary_path = latest_job_dir / 'dataset_generation' / 'dataset_generation_summary.json'
    if not summary_path.exists():
        print('Resume introuvable:', summary_path)
    else:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print('job_id               =', summary.get('job_id'))
        print('dataset_source_id    =', summary.get('dataset_source_id'))
        print('source_mode          =', summary.get('source_mode'))
        print('total_samples        =', summary.get('total_samples'))
        print('selected_samples     =', summary.get('selected_samples'))
        print('duplicate_samples    =', summary.get('duplicate_samples'))
        print('sampled_dir          =', summary.get('sampled_dir'))
        source_breakdown = summary.get('source_breakdown', {})
        if source_breakdown:
            print('\nSource breakdown:')
            ranked_items = sorted(source_breakdown.items(), key=lambda item: item[1].get('selected_samples', 0), reverse=True)
            for source_id, stats in ranked_items:
                print(
                    '-', source_id,
                    '| scanned_files=', stats.get('scanned_files', 0),
                    '| scanned_samples=', stats.get('scanned_samples', 0),
                    '| selected_files=', stats.get('selected_files', 0),
                    '| selected_samples=', stats.get('selected_samples', 0),
                    '| duplicate_samples=', stats.get('duplicate_samples', 0),
                    '| copied_raw_files=', stats.get('copied_raw_files', 0),
                )
        else:
            print('\nPas de source_breakdown pour ce mode de generation')


## 8. Construire le dataset final

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-build --config $DATASET_BUILD_CONFIG --job-id $DATASET_BUILD_JOB_ID $DATASET_BUILD_EXTRA_ARGS"

## 9. Fusionner des datasets finaux existants

In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main dataset-merge-final --config $DATASET_MERGE_FINAL_CONFIG --job-id $DATASET_MERGE_FINAL_JOB_ID $DATASET_MERGE_FINAL_EXTRA_ARGS"

Lecture rapide du dernier resume `dataset-merge-final`, avec affichage du `source_breakdown` par split et par dataset final source.

In [ ]:
from pathlib import Path
import json
import re

jobs_root = Path(DRIVE_ROOT) / 'jobs'
requested_job_id = DATASET_MERGE_FINAL_JOB_ID

match = re.match(r'^(.*?)(\d+)$', requested_job_id)
if match:
    prefix = match.group(1)
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and re.match(rf'^{re.escape(prefix)}\d+$', path.name)]
else:
    candidates = [path for path in jobs_root.iterdir() if path.is_dir() and path.name.startswith(requested_job_id)]

if not candidates:
    print('Aucun job dataset-merge-final trouve pour', requested_job_id)
else:
    latest_job_dir = sorted(candidates, key=lambda path: path.stat().st_mtime)[-1]
    summary_path = latest_job_dir / 'dataset_merge_final' / 'dataset_merge_final_summary.json'
    if not summary_path.exists():
        print('Resume introuvable:', summary_path)
    else:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print('job_id               =', summary.get('job_id'))
        print('dataset_id           =', summary.get('dataset_id'))
        print('build_mode           =', summary.get('build_mode'))
        print('teacher              =', f"{summary.get('teacher_engine')}:{summary.get('teacher_level')}")
        print('labeled_samples      =', summary.get('labeled_samples'))
        print('output_dir           =', summary.get('output_dir'))
        print('source_dataset_ids   =', summary.get('source_dataset_ids'))
        source_breakdown = summary.get('source_breakdown', {})
        if source_breakdown:
            for split_name, split_stats in source_breakdown.items():
                print(f'\nSplit: {split_name}')
                ranked_items = sorted(split_stats.items(), key=lambda item: item[1].get('kept_samples', 0), reverse=True)
                for source_id, stats in ranked_items:
                    print(
                        '-', source_id,
                        '| input_samples=', stats.get('input_samples', 0),
                        '| kept_samples=', stats.get('kept_samples', 0),
                        '| duplicate_samples=', stats.get('duplicate_samples', 0),
                        '| unique_games=', stats.get('unique_games', 0),
                    )
        else:
            print('\nPas de source_breakdown pour ce merge final')


## 10. Entrainement GPU

Previsualisation du dataset final le plus grand et du modele promu courant avant le training.

In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
promoted_metadata_path = Path(DRIVE_ROOT) / 'models' / 'promoted' / 'best' / 'metadata.json'

if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    candidates = []
    for item in registry.get('built_datasets', []):
        output_dir = Path(str(item.get('output_dir', '')))
        if (output_dir / 'train.npz').exists() and (output_dir / 'validation.npz').exists():
            candidates.append(item)
    if candidates:
        candidates = sorted(
            candidates,
            key=lambda item: (int(item.get('labeled_samples', 0)), str(item.get('updated_at', ''))),
            reverse=True,
        )
        best_dataset = candidates[0]
        print('Largest final dataset selected by training:')
        print('  dataset_id      =', best_dataset.get('dataset_id'))
        print('  build_mode      =', best_dataset.get('build_mode', 'teacher_label'))
        print('  labeled_samples =', best_dataset.get('labeled_samples'))
        print('  teacher         =', f"{best_dataset.get('teacher_engine')}:{best_dataset.get('teacher_level')}")
        print('  output_dir      =', best_dataset.get('output_dir'))
    else:
        print('Aucun built dataset complet avec train.npz + validation.npz trouve dans le registre')
else:
    print('dataset_registry.json introuvable:', registry_path)

if promoted_metadata_path.exists():
    promoted_metadata = json.loads(promoted_metadata_path.read_text(encoding='utf-8'))
    print('\nCurrent promoted best model:')
    print('  model_id               =', promoted_metadata.get('model_id'))
    print('  promoted_checkpoint    =', promoted_metadata.get('promoted_checkpoint_path'))
    print('  best_validation_metric =', promoted_metadata.get('best_validation_metric'))
    print('  benchmark_score        =', promoted_metadata.get('benchmark_score'))
else:
    print('\nAucun metadata de modele promu trouve:', promoted_metadata_path)


Comparer rapidement deux datasets finaux existants a partir du registre et de leur `dataset_metadata.json`.

In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
if not COMPARE_DATASET_ID_A or not COMPARE_DATASET_ID_B:
    print('Renseigne COMPARE_DATASET_ID_A et COMPARE_DATASET_ID_B pour lancer la comparaison')
elif not registry_path.exists():
    print('Aucun dataset_registry.json trouve:', registry_path)
else:
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    built_index = {
        str(item.get('dataset_id')): item
        for item in registry.get('built_datasets', [])
        if str(item.get('dataset_id', '')).strip()
    }

    def _load_metadata(dataset_id: str):
        entry = built_index.get(dataset_id)
        if not entry:
            return None, f'Dataset introuvable dans le registre: {dataset_id}'
        metadata_path = Path(str(entry['output_dir'])) / 'dataset_metadata.json'
        if not metadata_path.exists():
            return None, f'dataset_metadata.json introuvable pour {dataset_id}: {metadata_path}'
        return json.loads(metadata_path.read_text(encoding='utf-8')), None

    metadata_a, error_a = _load_metadata(COMPARE_DATASET_ID_A)
    metadata_b, error_b = _load_metadata(COMPARE_DATASET_ID_B)
    if error_a:
        print(error_a)
    if error_b:
        print(error_b)
    if metadata_a and metadata_b:
        print('Dataset A =', COMPARE_DATASET_ID_A)
        print('  build_mode   =', metadata_a.get('build_mode'))
        print('  teacher      =', f"{metadata_a.get('teacher_engine')}:{metadata_a.get('teacher_level')}")
        print('  labeled      =', metadata_a.get('labeled_samples'))
        print('  source_ids   =', metadata_a.get('source_dataset_ids', []))
        print('  output_dir   =', metadata_a.get('output_dir'))
        print('Dataset B =', COMPARE_DATASET_ID_B)
        print('  build_mode   =', metadata_b.get('build_mode'))
        print('  teacher      =', f"{metadata_b.get('teacher_engine')}:{metadata_b.get('teacher_level')}")
        print('  labeled      =', metadata_b.get('labeled_samples'))
        print('  source_ids   =', metadata_b.get('source_dataset_ids', []))
        print('  output_dir   =', metadata_b.get('output_dir'))

        print('\nSplit comparison:')
        all_splits = sorted(set(metadata_a.get('splits', {}).keys()) | set(metadata_b.get('splits', {}).keys()))
        for split_name in all_splits:
            split_a = metadata_a.get('splits', {}).get(split_name, {})
            split_b = metadata_b.get('splits', {}).get(split_name, {})
            samples_a = int(split_a.get('samples', 0))
            samples_b = int(split_b.get('samples', 0))
            games_a = int(split_a.get('games', 0))
            games_b = int(split_b.get('games', 0))
            print(
                '-', split_name,
                '| A samples=', samples_a,
                '| B samples=', samples_b,
                '| diff=', samples_b - samples_a,
                '| A games=', games_a,
                '| B games=', games_b,
            )

        if metadata_a.get('teacher_engine') == metadata_b.get('teacher_engine') and metadata_a.get('teacher_level') == metadata_b.get('teacher_level'):
            print('\nTeacher compatible: oui')
        else:
            print('\nTeacher compatible: non')

        merge_breakdown_b = metadata_b.get('merge_breakdown') or {}
        if merge_breakdown_b:
            print('\nMerge breakdown for dataset B:')
            for split_name, split_stats in merge_breakdown_b.items():
                print(
                    '-', split_name,
                    '| input_samples=', split_stats.get('input_samples', 0),
                    '| kept_samples=', split_stats.get('kept_samples', 0),
                    '| duplicate_samples=', split_stats.get('duplicate_samples', 0),
                    '| unique_games=', split_stats.get('unique_games', 0),
                )


In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main train --config $TRAIN_CONFIG --job-id $TRAIN_JOB_ID"

## 11. Evaluation

Previsualisation du dataset final le plus grand et du modele/checkpoint qui seront utilises par l'evaluation.

In [ ]:
from pathlib import Path
import json

registry_path = Path(DRIVE_ROOT) / 'data' / 'dataset_registry.json'
models_root = Path(DRIVE_ROOT) / 'models'
promoted_metadata_path = models_root / 'promoted' / 'best' / 'metadata.json'
model_registry_path = models_root / 'model_registry.json'

if registry_path.exists():
    registry = json.loads(registry_path.read_text(encoding='utf-8'))
    candidates = []
    for item in registry.get('built_datasets', []):
        output_dir = Path(str(item.get('output_dir', '')))
        if (output_dir / 'test.npz').exists():
            candidates.append(item)
    if candidates:
        candidates = sorted(
            candidates,
            key=lambda item: (int(item.get('labeled_samples', 0)), str(item.get('updated_at', ''))),
            reverse=True,
        )
        best_dataset = candidates[0]
        print('Largest final dataset selected by evaluation:')
        print('  dataset_id      =', best_dataset.get('dataset_id'))
        print('  build_mode      =', best_dataset.get('build_mode', 'teacher_label'))
        print('  labeled_samples =', best_dataset.get('labeled_samples'))
        print('  teacher         =', f"{best_dataset.get('teacher_engine')}:{best_dataset.get('teacher_level')}")
        print('  test_path       =', Path(str(best_dataset.get('output_dir', ''))) / 'test.npz')
    else:
        print('Aucun built dataset avec test.npz trouve dans le registre')
else:
    print('dataset_registry.json introuvable:', registry_path)

eval_model_mode = 'auto_latest'
if model_registry_path.exists():
    model_registry = json.loads(model_registry_path.read_text(encoding='utf-8'))
    models = model_registry.get('models', [])
    if models:
        models = sorted(models, key=lambda item: float(item.get('sort_ts', 0.0)), reverse=True)
        latest_model = models[0]
        print('\nCurrent latest model for evaluation auto_latest:')
        print('  model_id          =', latest_model.get('model_id'))
        print('  checkpoint_path   =', latest_model.get('checkpoint_path'))
        print('  evaluation_top1   =', latest_model.get('evaluation_top1'))
        print('  benchmark_score   =', latest_model.get('benchmark_score'))
    else:
        print('\nAucun modele present dans model_registry.json')
else:
    print('\nmodel_registry.json introuvable:', model_registry_path)

if promoted_metadata_path.exists():
    promoted_metadata = json.loads(promoted_metadata_path.read_text(encoding='utf-8'))
    print('\nCurrent promoted best model (utile si evaluation auto_best):')
    print('  model_id               =', promoted_metadata.get('model_id'))
    print('  promoted_checkpoint    =', promoted_metadata.get('promoted_checkpoint_path'))
    print('  best_validation_metric =', promoted_metadata.get('best_validation_metric'))
    print('  benchmark_score        =', promoted_metadata.get('benchmark_score'))


In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main evaluate --config $EVALUATION_CONFIG --job-id $EVALUATION_JOB_ID"

## 12. Benchmark

In [ ]:
from pathlib import Path
import json

try:
    import yaml
except ModuleNotFoundError:
    yaml = None

benchmark_config_path = Path(WORKTREE) / BENCHMARK_CONFIG
models_root = Path(DRIVE_ROOT) / 'models'
promoted_metadata_path = models_root / 'promoted' / 'best' / 'metadata.json'
model_registry_path = models_root / 'model_registry.json'

benchmark_cfg = {}
if yaml is None:
    print('PyYAML introuvable dans l'environnement notebook, impossible de lire la config benchmark automatiquement')
elif not benchmark_config_path.exists():
    print('Fichier de config benchmark introuvable:', benchmark_config_path)
else:
    raw_cfg = yaml.safe_load(benchmark_config_path.read_text(encoding='utf-8')) or {}
    benchmark_cfg = raw_cfg.get('benchmark', {}) or {}

target_mode = str(benchmark_cfg.get('target', 'auto_latest')) if benchmark_cfg else 'auto_latest'
checkpoint_cfg = str(benchmark_cfg.get('checkpoint_path', '')) if benchmark_cfg else ''
matchups = list(benchmark_cfg.get('matchups', [])) if benchmark_cfg else []
opponent_ratings = dict(benchmark_cfg.get('opponent_ratings', {})) if benchmark_cfg else {}
games_per_matchup = benchmark_cfg.get('games_per_matchup') if benchmark_cfg else None
alternate_first_player = benchmark_cfg.get('alternate_first_player') if benchmark_cfg else None
max_moves = benchmark_cfg.get('max_moves') if benchmark_cfg else None

print('Benchmark preview:')
print('  config_path             =', benchmark_config_path)
print('  target_mode             =', target_mode)
print('  checkpoint_path_cfg     =', checkpoint_cfg or '<auto>')
print('  matchups                =', ', '.join(matchups) if matchups else '<aucun>')
print('  games_per_matchup       =', games_per_matchup)
print('  alternate_first_player  =', alternate_first_player)
print('  max_moves               =', max_moves)
if opponent_ratings:
    print('  opponent_ratings        =')
    for key in matchups:
        print('    -', key, '=>', opponent_ratings.get(key, '<default>'))

latest_model = None
if model_registry_path.exists():
    model_registry = json.loads(model_registry_path.read_text(encoding='utf-8'))
    models = model_registry.get('models', [])
    if models:
        models = sorted(models, key=lambda item: float(item.get('sort_ts', 0.0)), reverse=True)
        latest_model = models[0]

if latest_model:
    print('\nCurrent latest model for benchmark auto_latest:')
    print('  model_id                =', latest_model.get('model_id'))
    print('  latest_checkpoint       =', latest_model.get('latest_checkpoint_path'))
    print('  benchmark_score         =', latest_model.get('benchmark_score'))
    print('  benchmark_elo_estimate  =', latest_model.get('benchmark_elo_estimate'))
    print('  benchmark_history_path  =', latest_model.get('benchmark_history_path'))
else:
    print('\nAucun modele disponible dans model_registry.json pour auto_latest')

promoted_metadata = None
if promoted_metadata_path.exists():
    promoted_metadata = json.loads(promoted_metadata_path.read_text(encoding='utf-8'))
    print('\nCurrent promoted best model:')
    print('  model_id               =', promoted_metadata.get('model_id'))
    print('  promoted_checkpoint    =', promoted_metadata.get('promoted_checkpoint_path'))
    print('  best_validation_metric =', promoted_metadata.get('best_validation_metric'))
    print('  benchmark_score        =', promoted_metadata.get('benchmark_score'))
else:
    print('\nAucun metadata de modele promu trouve:', promoted_metadata_path)

print('\nBenchmark target resolved preview:')
if target_mode in {'engine_v1'} or checkpoint_cfg == 'engine_v1':
    print('  resolved_target        = engine_v1')
elif target_mode in {'auto_best', 'auto_promoted_best'} or checkpoint_cfg in {'auto_best', 'auto_promoted_best'}:
    if promoted_metadata:
        print('  resolved_target        = auto_best / auto_promoted_best')
        print('  resolved_model_id      =', promoted_metadata.get('model_id'))
        print('  resolved_checkpoint    =', models_root / 'promoted' / 'best' / 'model.pt')
    else:
        print('  resolved_target        = auto_best / auto_promoted_best')
        print('  resolved_checkpoint    = indisponible, aucun modele promu trouve')
elif target_mode in {'auto', 'auto_latest'} or checkpoint_cfg in {'', 'auto', 'auto_latest'}:
    if latest_model:
        print('  resolved_target        = auto_latest')
        print('  resolved_model_id      =', latest_model.get('model_id'))
        print('  resolved_checkpoint    =', latest_model.get('latest_checkpoint_path'))
    else:
        print('  resolved_target        = auto_latest')
        print('  resolved_checkpoint    = indisponible, aucun modele dans le registre')
else:
    print('  resolved_target        = checkpoint explicite')
    print('  resolved_checkpoint    =', checkpoint_cfg)


In [ ]:
!bash -lc "cd $WORKTREE && PYTHONPATH=src python -m songo_model_stockfish.cli.main benchmark --config $BENCHMARK_CONFIG --job-id $BENCHMARK_JOB_ID"

In [ ]:
from pathlib import Path
import json
from datetime import datetime

reports_root = Path(DRIVE_ROOT) / 'reports' / 'benchmarks'
models_root = Path(DRIVE_ROOT) / 'models'
history_path = models_root / 'history' / 'benchmark_history.jsonl'
job_prefix = BENCHMARK_JOB_ID.rsplit('_v', 1)[0] if '_v' in BENCHMARK_JOB_ID else BENCHMARK_JOB_ID
job_dirs = sorted(
    [path for path in reports_root.glob(f'{job_prefix}*') if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

def _fmt_ts(value):
    if value in (None, ''):
        return '<unknown>'
    try:
        return datetime.fromtimestamp(float(value)).strftime('%Y-%m-%d %H:%M:%S')
    except Exception:
        return str(value)

if not job_dirs:
    print('Aucun job benchmark trouve pour le prefixe:', job_prefix)
else:
    selected_job_dir = job_dirs[0]
    summary_path = selected_job_dir / 'benchmark' / 'benchmark_summary.json'
    if not summary_path.exists():
        print('benchmark_summary.json introuvable:', summary_path)
    else:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        matchups = list(summary.get('matchups', []))
        print('Benchmark summary preview:')
        print('  job_dir                  =', selected_job_dir)
        print('  summary_path             =', summary_path)
        print('  engine                   =', summary.get('engine'))
        print('  matchups_count           =', len(matchups))
        print('  benchmark_score          =', summary.get('benchmark_score'))
        print('  benchmark_score_weighted =', summary.get('benchmark_score_weighted'))
        print('  benchmark_elo_estimate   =', summary.get('benchmark_elo_estimate'))
        print('  benchmark_history_path   =', summary.get('benchmark_history_path'))

        history_rows = []
        if history_path.exists() and summary.get('engine'):
            for line in history_path.read_text(encoding='utf-8').splitlines():
                line = line.strip()
                if not line:
                    continue
                payload = json.loads(line)
                if payload.get('model_id') == summary.get('engine'):
                    history_rows.append(payload)

        if history_rows:
            latest_history = history_rows[-1]
            print('\nLatest benchmark history entry for this model:')
            print('  model_id                =', latest_history.get('model_id'))
            print('  benchmark_score         =', latest_history.get('benchmark_score'))
            print('  benchmark_score_weighted=', latest_history.get('benchmark_score_weighted'))
            print('  benchmark_elo_estimate  =', latest_history.get('benchmark_elo_estimate'))
            print('  job_id                  =', latest_history.get('job_id'))
            print('  benchmark_summary_path  =', latest_history.get('benchmark_summary_path'))

            print('\nRecent benchmark history (last 8 runs):')
            print('  created_at           | job_id                       | weighted  | elo     | score')
            print('  ------------------- | ---------------------------- | --------- | ------- | -------')
            for row in history_rows[-8:]:
                created_at = _fmt_ts(row.get('created_at'))
                job_id = str(row.get('job_id', ''))[:28].ljust(28)
                weighted = f"{float(row.get('benchmark_score_weighted', -1.0)):.4f}"
                elo = row.get('benchmark_elo_estimate')
                elo_str = f"{float(elo):.1f}" if elo not in (None, '') else '<na>'
                score = f"{float(row.get('benchmark_score', -1.0)):.4f}"
                print(f'  {created_at} | {job_id} | {weighted:>9} | {elo_str:>7} | {score:>7}')

        if matchups:
            sorted_matchups = sorted(matchups, key=lambda item: float(item.get('winrate', 0.0)), reverse=True)
            avg_winrate = sum(float(item.get('winrate', 0.0)) for item in sorted_matchups) / len(sorted_matchups)
            total_games = sum(int(item.get('games', 0)) for item in sorted_matchups)
            total_wins = sum(int(item.get('wins_a', 0)) for item in sorted_matchups)
            total_losses = sum(int(item.get('wins_b', 0)) for item in sorted_matchups)
            total_draws = sum(int(item.get('draws', 0)) for item in sorted_matchups)
            print('  average_winrate          =', round(avg_winrate, 4))
            print('  total_games              =', total_games)
            print('  total_record             =', f'{total_wins}W / {total_losses}L / {total_draws}D')
            print('\nPer-matchup results:')
            for item in sorted_matchups:
                opponent = item.get('opponent')
                games = int(item.get('games', 0))
                wins = int(item.get('wins_a', 0))
                losses = int(item.get('wins_b', 0))
                draws = int(item.get('draws', 0))
                winrate = float(item.get('winrate', 0.0))
                score_rate = float(item.get('score_rate', 0.0))
                avg_moves = float(item.get('avg_moves', 0.0))
                weighted = winrate * float(item.get('difficulty_weight', 1.0))
                opp_rating = float(item.get('opponent_rating', 0.0))
                as_first = item.get('as_first_player', {})
                as_second = item.get('as_second_player', {})
                print(
                    f'  - {opponent}: {wins}W / {losses}L / {draws}D | games={games} | winrate={winrate:.4f} | score_rate={score_rate:.4f} | weighted={weighted:.4f} | opp_rating={opp_rating:.0f} | avg_moves={avg_moves:.1f}'
                )
                print(
                    f"    as_first={as_first.get('wins', 0)}W/{as_first.get('losses', 0)}L/{as_first.get('draws', 0)}D | winrate={float(as_first.get('winrate', 0.0)):.4f} | score_rate={float(as_first.get('score_rate', 0.0)):.4f}"
                )
                print(
                    f"    as_second={as_second.get('wins', 0)}W/{as_second.get('losses', 0)}L/{as_second.get('draws', 0)}D | winrate={float(as_second.get('winrate', 0.0)):.4f} | score_rate={float(as_second.get('score_rate', 0.0)):.4f}"
                )
        else:
            print('Aucun matchup present dans le summary benchmark')


In [ ]:
from pathlib import Path
import json
from datetime import datetime

models_root = Path(DRIVE_ROOT) / 'models'
model_registry_path = models_root / 'model_registry.json'
history_path = models_root / 'history' / 'benchmark_history.jsonl'

def _fmt_ts(value):
    if value in (None, ''):
        return '<unknown>'
    try:
        return datetime.fromtimestamp(float(value)).strftime('%Y-%m-%d %H:%M:%S')
    except Exception:
        return str(value)

def _load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

if not model_registry_path.exists():
    print('model_registry.json introuvable:', model_registry_path)
else:
    registry = json.loads(model_registry_path.read_text(encoding='utf-8'))
    models = registry.get('models', [])
    if not models:
        print('Aucun modele present dans le registre')
    else:
        models = sorted(models, key=lambda item: float(item.get('sort_ts', 0.0)), reverse=True)
        current = models[0]
        model_id = str(current.get('model_id', ''))
        print('Unified latest model cycle view:')
        print('  model_id                =', model_id)
        print('  training_job_id         =', current.get('training_job_id'))
        print('  best_validation_metric  =', current.get('best_validation_metric'))
        print('  evaluation_top1         =', current.get('evaluation_top1'))
        print('  benchmark_score         =', current.get('benchmark_score'))
        print('  benchmark_elo_estimate  =', current.get('benchmark_elo_estimate'))
        print('  model_card_path         =', current.get('model_card_path'))

        model_card_path = Path(str(current.get('model_card_path', ''))) if current.get('model_card_path') else None
        model_card = _load_json(model_card_path) if model_card_path else None

        training_summary = None
        evaluation_summary = None
        if model_card:
            training_job_id = str(model_card.get('training_job_id', '')).strip()
            if training_job_id:
                training_summary = _load_json(Path(DRIVE_ROOT) / 'jobs' / training_job_id / 'training_summary.json')
            eval_summary_path = str(model_card.get('evaluation_summary_path', '')).strip()
            if eval_summary_path:
                evaluation_summary = _load_json(Path(eval_summary_path))

        if training_summary:
            print('\nTraining snapshot:')
            print('  dataset_id              =', training_summary.get('dataset_id'))
            print('  selection_mode          =', training_summary.get('dataset_selection_mode'))
            print('  best_epoch              =', training_summary.get('best_epoch'))
            print('  best_validation_metric  =', training_summary.get('best_validation_metric'))
            print('  early_stopped           =', training_summary.get('early_stopped'))
            print('  final_model_path        =', training_summary.get('final_model_path'))
        else:
            print('\nTraining snapshot: introuvable')

        if evaluation_summary:
            print('\nLatest evaluation snapshot:')
            print('  dataset_id              =', evaluation_summary.get('dataset_id'))
            print('  selection_mode          =', evaluation_summary.get('dataset_selection_mode'))
            print('  policy_accuracy_top1    =', evaluation_summary.get('policy_accuracy_top1'))
            print('  policy_accuracy_top3    =', evaluation_summary.get('policy_accuracy_top3'))
            print('  value_mae               =', evaluation_summary.get('value_mae'))
            print('  loss_total              =', evaluation_summary.get('loss_total'))
            print('  evaluation_summary_path =', model_card.get('evaluation_summary_path'))
        else:
            print('\nLatest evaluation snapshot: introuvable')

        history_rows = []
        if history_path.exists():
            for line in history_path.read_text(encoding='utf-8').splitlines():
                line = line.strip()
                if not line:
                    continue
                payload = json.loads(line)
                if payload.get('model_id') == model_id:
                    history_rows.append(payload)

        if history_rows:
            print('\nUnified recent cycle table (last 8 benchmark cycles for this model):')
            print('  created_at           | job_id                       | weighted  | elo     | eval_top1 | best_val')
            print('  ------------------- | ---------------------------- | --------- | ------- | --------- | --------')
            eval_top1 = current.get('evaluation_top1')
            best_val = current.get('best_validation_metric')
            eval_str = f"{float(eval_top1):.4f}" if eval_top1 not in (None, '', -1.0) else '<na>'
            best_val_str = f"{float(best_val):.4f}" if best_val not in (None, '', -1.0) else '<na>'
            for row in history_rows[-8:]:
                created_at = _fmt_ts(row.get('created_at'))
                job_id = str(row.get('job_id', ''))[:28].ljust(28)
                weighted = f"{float(row.get('benchmark_score_weighted', -1.0)):.4f}"
                elo = row.get('benchmark_elo_estimate')
                elo_str = f"{float(elo):.1f}" if elo not in (None, '') else '<na>'
                print(f'  {created_at} | {job_id} | {weighted:>9} | {elo_str:>7} | {eval_str:>9} | {best_val_str:>8}')
        else:
            print('\nUnified recent cycle table: aucun historique benchmark trouve pour ce modele')


## 11bis. Lire les resultats de comparaison

Ordre de lecture recommande:
- 1. `benchmark_score` et winrates benchmark
- 2. `policy_accuracy_top1`, `policy_accuracy_top3`, `value_mae`
- 3. `best_validation_metric`

Regle simple:
- si `residual` est meilleur en benchmark et pas moins bon en evaluation, il gagne
- s'il est meilleur en evaluation mais pas en benchmark, garde `stable`

Fichiers a comparer pour chaque variante:
- `jobs/<train_job_id>/training_summary.json`
- `jobs/<eval_job_id>/evaluation_summary.json`
- `jobs/<benchmark_job_id>/benchmark_summary.json`


## 12. Suivre un job en direct

In [ ]:
WATCH_JOB_ID = TRAIN_JOB_ID
!tail -f {DRIVE_ROOT}/jobs/{WATCH_JOB_ID}/events.jsonl

## 13. Reprendre un job interrompu

Relance simplement la meme cellule de commande avec le meme `job_id`.